# 02 — Model Benchmarking

Compare detection backends (YOLOv11 vs YOLOv9 vs RT-DETR) on latency and mAP.
Requires the `training` extra and downloaded validation data.

In [ ]:
import time

import numpy as np
import pandas as pd
from ultralytics import YOLO

MODELS = {
    "YOLOv11n": "yolo11n.pt",
    "YOLOv11s": "yolo11s.pt",
    "YOLOv9c": "yolov9c.pt",
    "RT-DETR": "rtdetr-l.pt",
}
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

In [ ]:
def bench(weights, iters=30):
    model = YOLO(weights)
    for _ in range(3):
        model.predict(dummy, verbose=False)  # warmup
    t = []
    for _ in range(iters):
        s = time.perf_counter()
        model.predict(dummy, verbose=False)
        t.append((time.perf_counter() - s) * 1000)
    return float(np.mean(t)), float(1000 / np.mean(t))


rows = []
for name, w in MODELS.items():
    try:
        ms, fps = bench(w)
        rows.append({"model": name, "latency_ms": round(ms, 1), "fps": round(fps, 1)})
    except Exception as e:
        rows.append({"model": name, "latency_ms": None, "fps": None, "error": str(e)})
df = pd.DataFrame(rows)
df

In [ ]:
import matplotlib.pyplot as plt

valid = df.dropna(subset=["fps"])
if not valid.empty:
    valid.plot.bar(x="model", y="fps", title="Throughput (FPS) by backend", legend=False)
    plt.ylabel("FPS")
    plt.show()